# <center>Big Data &ndash; Exercises</center>
## <center>Fall 2021 &ndash; Week 9 &ndash; ETH Zurich</center>
## <center>Spark Dataframes and Spark SQL, Moodle exercise</center>

# Preparation for the moodle exercise in Spark

In this jupyter notebook we are going to make the preprocessing part of the dataset that is going to be used in the graded exercise of this week.
It will be the same language game dataset as in exercise08.

1. Change to exercise09 repository

2. Start docker <br>
```docker-compose up -d```

3. Getting the data:
Follow the procedure that is described below. The dataset can be found here: http://data.greatlanguagegame.com.s3.amazonaws.com/confusion-2014-03-02.tbz2. 

More specifically do the following:
- download the data      :<br> ```wget http://data.greatlanguagegame.com.s3.amazonaws.com/confusion-2014-03-02.tbz2```
- extract the data       :<br> ```tar -jxvf confusion-2014-03-02.tbz2```

4. copy the data to docker :<br> ```docker cp confusion-2014-03-02/confusion-2014-03-02.json jupyter:/home/jovyan/work``` <br>
(Copying the data to docker needs to be done only once and it might take 1-2 minutes.)

## More Info about the data
You can find more information about the dataset (as well as the schema and examples) in this link: http://lars.yencken.org/datasets/languagegame/

## Instructions:

In every query we ask you for three quantities: the query itself, the result of the query as well as the productivity time. That means the development time of each query (time elapsed before you start writing the query, and the time at which the correct, final query is ready). Note that the time part of every question is optional and not graded. In order to make easier the time recording we created two functions that do it automatically. Run the cell below in order to import the functions into the current notebook. Then before each query we will have a ```start_exercise()``` cell that you have to run in order to start time recording. After you have finished your query and you are sure about the answer run the ```finish_exercise()``` one to get the time measurement. 

In [184]:
import time

def start_exercise():
    global last
    last = time.time()
    
def finish_exercise():
    global last
    print("This exercise took {0}s".format(int(time.time()-last)))

## <center>1. Spark Dataframes</center>

Write queries for the same questions as last week, but this time using Spark Dataframes operations (the data loading will take a minute)

### 1.0. Data preprocessing

In [185]:
import json
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark = SparkSession.builder.master('local').getOrCreate()
sc = spark.sparkContext

path = "confusion-2014-03-02.json"
dataset = spark.read.json(path).cache()

In [186]:
#test it out
dataset.limit(3).show()

+--------------------+-------+----------+---------+--------------------+---------+
|             choices|country|      date|    guess|              sample|   target|
+--------------------+-------+----------+---------+--------------------+---------+
|[Maori, Mandarin,...|     AU|2013-08-19|Norwegian|48f9c924e0d98c959...|Norwegian|
|[Danish, Dinka, K...|     AU|2013-08-19|    Dinka|af5e8f27cef9e689a...|    Dinka|
|[German, Hungaria...|     AU|2013-08-19|  Turkish|509c36eb58dbce009...|   Samoan|
+--------------------+-------+----------+---------+--------------------+---------+



## Assignment 1
Find the number of games where the guessed language is correct (meaning equal to the target one) and that language is Russian.

In [187]:
start_exercise()

In [188]:
dataset.printSchema()

root
 |-- choices: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- country: string (nullable = true)
 |-- date: string (nullable = true)
 |-- guess: string (nullable = true)
 |-- sample: string (nullable = true)
 |-- target: string (nullable = true)



In [189]:
#Your code here
dataset1 = dataset.filter("target = guess")
dataset1 = dataset1.filter(dataset1.guess == 'Russian')
dataset1.show()
dataset1.count()

+--------------------+-------+----------+-------+--------------------+-------+
|             choices|country|      date|  guess|              sample| target|
+--------------------+-------+----------+-------+--------------------+-------+
|[Lao, Nepali, Rus...|     AU|2013-08-19|Russian|8a59d48e99e8a1df7...|Russian|
|[Arabic, Estonian...|     AU|2013-08-19|Russian|8a59d48e99e8a1df7...|Russian|
|[Croatian, Nepali...|     AU|2013-08-20|Russian|8a59d48e99e8a1df7...|Russian|
|[Gujarati, Malaya...|     AU|2013-08-20|Russian|8a59d48e99e8a1df7...|Russian|
|[Cantonese, Fijia...|     AU|2013-08-20|Russian|8a59d48e99e8a1df7...|Russian|
|[Armenian, Russia...|     AU|2013-08-20|Russian|8a59d48e99e8a1df7...|Russian|
|[Arabic, Kurdish,...|     AU|2013-08-20|Russian|ff5b5a0d34c77d2e2...|Russian|
|[Indonesian, Ital...|     AU|2013-08-20|Russian|b7df3f9d67cef259f...|Russian|
|[Arabic, Hebrew, ...|     AU|2013-08-20|Russian|8a59d48e99e8a1df7...|Russian|
|[Danish, Kannada,...|     AU|2013-08-20|Russian|ff5

290818

In [190]:
finish_exercise()

This exercise took 2s


## Assignment 2
Return the number of distinct "target" languages.

In [191]:
start_exercise()

In [192]:
#Your code here
dataset2 = dataset.select("target")
dataset2.groupBy("target")
dataset2.distinct().count()

78

In [193]:
finish_exercise()

This exercise took 3s


## Assignment 3
Return the sample IDs (i.e., the *sample* field) of the top three games where the guessed language is correct (equal to the target one) ordered by language (ascending), then by country (ascending), then by date (ascending).

In [194]:
start_exercise()

In [195]:
#Your code here
dataset3 = dataset.select("*")
dataset3 = dataset3.filter('target = guess')
dataset3 = dataset3.orderBy(["target", "country", "date"], ascending=[1,1,1])
dataset3.select("sample").limit(10).collect() 
# dataset3.select("sample").distinct().limit(10).collect() ?distinct() sort agian?

[Row(sample='00b85faa8b878a14f8781be334deb137'),
 Row(sample='efcd813daec1c836d9f030b30caa07ce'),
 Row(sample='efcd813daec1c836d9f030b30caa07ce'),
 Row(sample='13722ceed1eede7ba597ade9b4cb9807'),
 Row(sample='1dd8e1883037c6305b87afe382c4feba'),
 Row(sample='efcd813daec1c836d9f030b30caa07ce'),
 Row(sample='1dd8e1883037c6305b87afe382c4feba'),
 Row(sample='fdf23d0a7063ba2fcef4b18eb7d57ad8'),
 Row(sample='00b85faa8b878a14f8781be334deb137'),
 Row(sample='efcd813daec1c836d9f030b30caa07ce')]

In [196]:
finish_exercise()

This exercise took 8s


## Assignment 4
Aggregate all games by country and target language, counting the number of guesses for each group and return the frequencies of the three most frequent country/language combinations.

In [197]:
start_exercise()

In [198]:
#Your code here
dataset4 = dataset.select("*")
dataset4 = dataset4.groupBy("target", "country").count()
dataset4 = dataset4.sort('count', ascending = False)
dataset4.select('count').limit(3).collect()

[Row(count=112934), Row(count=112007), Row(count=110919)]

In [199]:
finish_exercise()

This exercise took 6s


## Assignment 5
Find the percentage of games where (the answer was correct && the correct guess was the first choice amongst the array of possible answers)

Please write the fraction rounding to 4 decimals (eg. 0.3323)

In [200]:
start_exercise()

In [201]:
#Your code here
dataset5 = dataset.select("*")
dataset5 = dataset5.filter("target = guess")
dataset5 = dataset5.filter(dataset5.target == dataset5.choices[0])
print(dataset5.count()/dataset.count())

0.25603983084476356


In [202]:
finish_exercise()

This exercise took 4s


## Assignment 6
Sort the languages by decreasing overall percentage of correct guesses and return the first three languages.

In [203]:
start_exercise()

In [204]:
#Your code here
dataset6 = dataset.select('*')
dataset_total_count = dataset6.groupBy('target').count().withColumnRenamed('count', 'total')
# dataset_total_count = dataset6.groupBy('target').agg(count('target')).alias('total')
# dataset_total_count = dataset6.groupBy('target').count().alias('total')
dataset_total_count.show()
dataset_correct_count = dataset6.filter('target = guess' ).groupBy('target').count().withColumnRenamed('count', 'correct')
dataset_correct_count.show()
dataset_new = dataset_total_count.join(dataset_correct_count, "target")
dataset_new = dataset_new.select("*", (dataset_new.correct/dataset_new.total).alias('percent'))
dataset_new = dataset_new.orderBy('percent', ascending = False)
dataset_new.limit(3).show()

+----------------+------+
|          target| total|
+----------------+------+
|        Tigrinya|125291|
|          Bangla|136898|
|            Urdu|165659|
|          Fijian|115852|
|           Khmer|210424|
|         Maltese|131677|
|       Sinhalese|127123|
|          Nepali|137450|
|         Turkish|171309|
|        Armenian|184690|
|         Kannada|119509|
|         Finnish|244784|
|           Malay|135407|
|            Thai|306864|
|       Icelandic|193143|
|          Somali|194808|
|      Indonesian|118321|
|Northern Ndebele|108269|
|       Cantonese|315753|
|       Ukrainian|305112|
+----------------+------+
only showing top 20 rows

+----------------+-------+
|          target|correct|
+----------------+-------+
|          Bangla|  70685|
|        Tigrinya|  57049|
|            Urdu|  94480|
|           Khmer| 132036|
|          Fijian|  48081|
|         Maltese|  63216|
|       Sinhalese|  65010|
|          Nepali|  70924|
|         Turkish| 104322|
|        Armenian| 118356|

In [205]:
finish_exercise()

This exercise took 20s


## Assignment 7
Return the number of games played on the latest day.

In [206]:
start_exercise()

In [207]:
#Your code here
from pyspark.sql.functions import desc
dataset7 = dataset.select('*')
dataset7.orderBy(desc('date')).show()
dataset7 = dataset7.filter(dataset7.date == "2014-03-01")
dataset7.count()

+--------------------+-------+----------+----------+--------------------+----------+
|             choices|country|      date|     guess|              sample|    target|
+--------------------+-------+----------+----------+--------------------+----------+
| [Greek, Portuguese]|     US|2014-03-01|Portuguese|17527de743e8a5b5c...|     Greek|
|   [Gujarati, Hausa]|     RO|2014-03-01|     Hausa|56ecf19cff578aa76...|  Gujarati|
|[Hungarian, Serbian]|     GB|2014-03-01|   Serbian|026e4dc4264238c4c...| Hungarian|
|[Assyrian, Vietna...|     DE|2014-03-01|Vietnamese|12f884fce08164e9d...|Vietnamese|
|   [German, Tagalog]|     CA|2014-03-01|    German|1f8b9a59cd75fc429...|    German|
|  [Burmese, Tagalog]|     RU|2014-03-01|   Burmese|7b8d176552123a50d...|   Burmese|
|[Icelandic, Yiddish]|     NZ|2014-03-01|   Yiddish|1461114cca35b9334...| Icelandic|
|  [Latvian, Swahili]|     GB|2014-03-01|   Latvian|1b959fe55794f6d4e...|   Swahili|
| [Czech, Portuguese]|     NO|2014-03-01|     Czech|9769bab1b0b25

65653

In [208]:
finish_exercise()

This exercise took 10s


## <center>2. Spark SQL</center>

Write Spark SQL queries for the same questions as earlier.

### 2.0. Data preprocessing

In [209]:
!pip install sparksql-magic

In [210]:
%load_ext sparksql_magic

The sparksql_magic extension is already loaded. To reload it, use:
  %reload_ext sparksql_magic


In [211]:
path = "confusion-2014-03-02.json"
dataset = spark.read.json(path).cache()
dataset.registerTempTable("dataset")

In [212]:
%%sparksql
-- test it out
SELECT *
FROM dataset
LIMIT 3

choices,country,date,guess,sample,target
"['Maori', 'Mandarin', 'Norwegian', 'Tongan']",AU,2013-08-19,Norwegian,48f9c924e0d98c959d8a6f1862b3ce9a,Norwegian
"['Danish', 'Dinka', 'Khmer', 'Lao']",AU,2013-08-19,Dinka,af5e8f27cef9e689a070b8814dcc02c3,Dinka
"['German', 'Hungarian', 'Samoan', 'Turkish']",AU,2013-08-19,Turkish,509c36eb58dbce009ccf93f375358d53,Samoan


## Assignment 1
Find the number of games where the guessed language is correct (meaning equal to the target one) and that language is Russian.

In [213]:
start_exercise()

In [214]:
%%sparksql

SELECT count(target)
From dataset
Where target = guess AND target = 'Russian';



count(target)
290818


In [215]:
finish_exercise()

This exercise took 4s


## Assignment 2
Return the number of distinct "target" languages.

In [216]:
start_exercise()

In [217]:
%%sparksql
SELECT COUNT(TARGET)
FROM (
    Select DISTINCT target
    From dataset
)

count(TARGET)
78


In [218]:
finish_exercise()

This exercise took 7s


## Assignment 3
Return the sample IDs (i.e., the *sample* field) of the top three games where the guessed language is correct (equal to the target one) ordered by language (ascending), then by country (ascending), then by date (ascending).

In [219]:
start_exercise()

In [220]:
%%sparksql
    
Select * 
From dataset
WHERE target = guess
ORDER BY target , country , date 
limit 5

/*
SELECT DISTINCT sample
FROM (
    Select * 
    From dataset
    WHERE target = guess
    ORDER BY target , country , date 
 )
limit 3
*/


choices,country,date,guess,sample,target
"['Albanian', 'Macedonian']",A1,2013-09-04,Albanian,00b85faa8b878a14f8781be334deb137,Albanian
"['Albanian', 'Bulgarian', 'Indonesian', 'Portuguese']",A1,2013-09-05,Albanian,efcd813daec1c836d9f030b30caa07ce,Albanian
"['Albanian', 'Tamil']",A1,2013-09-08,Albanian,efcd813daec1c836d9f030b30caa07ce,Albanian
"['Albanian', 'Hindi', 'Swahili']",A1,2013-09-08,Albanian,13722ceed1eede7ba597ade9b4cb9807,Albanian
"['Albanian', 'Nepali']",A1,2013-09-08,Albanian,1dd8e1883037c6305b87afe382c4feba,Albanian


In [221]:
finish_exercise()

This exercise took 15s


## Assignment 4
Aggregate all games by country and target language, counting the number of guesses for each group and return the frequencies of the three most frequent country/language combinations.

In [222]:
start_exercise()

In [223]:
%%sparksql
SELECT target, country, count(target)
FROM dataset
GROUP BY target, country
ORDER BY count(target) DESC
LIMIT 3

target,country,count(target)
French,US,112934
German,US,112007
Spanish,US,110919


In [224]:
finish_exercise()

This exercise took 8s


## Assignment 5
Find the percentage of games where (the answer was correct && the correct guess was the first choice amongst the array of possible answers)

Please write the fraction rounding to 4 decimals (eg. 0.3323)

In [225]:
start_exercise()

In [226]:
%%sparksql

select count_one, count_two, count_one/count_two as percentage
FROM (
    SELECT count(*) AS count_one 
    FROM dataset
    WHERE target = guess and target = choices[0]
),
(
    SELECT count(*) AS count_two 
    FROM dataset
)





count_one,count_two,percentage
4227531,16511224,0.25603983084476356


In [227]:
finish_exercise()

This exercise took 5s


## Assignment 6
Sort the languages by decreasing overall percentage of correct guesses and return the first three languages.

In [228]:
start_exercise()

In [229]:
%%sparksql

With t1 as
(SELECT target, count(*) as total
 from dataset
GROUP by target),

t2 as
(SELECT target, count(*) as correct
FROM dataset
WHERE guess = target
GROUP By target)

Select * , (correct/total) as perc
from t1 
join t2 using (target)
Order By perc DESC


only showing top 20 row(s)


target,total,correct,perc
French,346851,325430,0.9382414927447232
German,342774,315271,0.9197634593055483
Spanish,339585,304147,0.8956432115670598
Italian,337258,300881,0.8921389559328466
Russian,331378,290818,0.877602013410667
Korean,326076,283090,0.8681718372403979
Japanese,327769,282007,0.8603833797583054
Mandarin,323878,278381,0.8595242653097771
Vietnamese,317797,267440,0.8415435010399721
Cantonese,315753,262619,0.8317228973279747


In [230]:
finish_exercise()

This exercise took 13s


## Assignment 7
Return the number of games played on the latest day.

In [231]:
start_exercise()

In [232]:
%%sparksql
/*
SELECT *
FROM dataset
ORDER BY date Desc;
*/
SELECT COUNT(*)
FROM (
    SELECT * 
    FROM dataset
    WHERE date = '2014-03-01'
)

count(1)
65653


In [233]:
finish_exercise()

This exercise took 2s
